# Metasyn multiple table tutorial

In this tutorial you will learn how to create synthetic versions of multiple tables at onc, while preserving some relations between tables.

First you should install metasyn if you have not done so already.

In [ ]:
# %pip install metasyn

### Loading the dataset

We will use a demonstration dataset that is built into metasyn, called `ShopMultiDataset`. This dataset contains three tables that are interrelated through customer id's and product id's.



In [ ]:
from metasyn.demo.dataset import ShopMultiDataset
from metasyn.multiframe import MultiFrame, ColumnRelation
from metasyn.builder import MultiFrameBuilder

Here we load in our demo dataset. In your own usecase, you simply need to read the files into
a dictionary of data frames.
For example: ``data = {"name_1": pl.read_csv("file1.csv"), ...}``

In [ ]:
data = ShopMultiDataset().get_dataframes()
print({key: d.head(1) for key, d in data.items()})

The tables have some relations between them. For example in the 'purchases' table we have the 'customer_id' column which has identifiers on who purchased that particular item. This 'customer_id' is also present in the 'customers' table. In a relational database, this is called a primary <-> foreign key relationship. This can be very useful when data from different tables have to be combined. Metasyn includes a multitable feature to capture these kinds of relations.

To examplify this, let's perform a simple join to combine the purchases and customers table:

In [ ]:
data["purchases"].join(data["customers"], left_on="customer_id", right_on="id")

### Synthesizing unrelated tables

Now, let us naively generate synthetic data independently using metasyn without specifying any relations.

In [ ]:
multiframe = MultiFrame.fit_dataframes(data, relations=[])
syn_data = multiframe.synthesize()
# Try to join the same tables
syn_data["purchases"].join(syn_data["customers"], left_on="customer_id", right_on="id")

Note above that while the synthetic data has the same number of rows for the tables, the number of rows in the joined table is vastly different. This is because of the fact that the customer identifiers in the two tables are created independently.

### Synthesizing related tables
To remedy this, we can specify relations in the dataset.

In [ ]:
relations = [
    "purchases[customer_id] SUBSET OF customers[id]",
    "purchases[product_id] SUBSET OF products[id]",
]

mfb = MultiFrameBuilder()

for key, df in data.items():
    mfb.add_dataframe(name=key, df=df)

for relation in relations:
    mfb.add_relation(ColumnRelation.parse(relation))

In the presented table we only have SUBSET OF relations, but there are a few more:

- `SUBSET OF`: Column a has values that are present in column b and can occur multiple times in column a.
- `EQUALS`: Column a has the same values as column b, but not necessarily in the same order. This also implies that the table of column a and the table of column b have the same number of rows.
- `EQUAL ORDERED`: Column a has the same values as column b and also in the same order. Also implies the tables have the same number of rows.
- `INFER FROM`: The relation between column a and b should be inferred by metasyn. This will result in one of the above relationships.

We can also adjust the size of the output tables for each individual table:

In [ ]:
multiframe_improved = mfb.fit()

rows = {
    'customers': 5, 
    'products': 10, 
    'purchases': 20, 
}

multiframe_improved.synthesize(n=rows)

### Inspecting multiframes

You can inspect multiframes with a print statement.

In [ ]:
print(multiframe_improved)

You can select and inspect metaframes (representations of the individual tables) with brackets `[]`:

In [ ]:
print(multiframe_improved["purchases"])

### Saving and loading multiframes

Similar to metaframes, multiframes can also be saved and loaded from a .json file. This .json file is a GMF (Generative Metadata Format) file that has the same structure as when the metadata of single tables are stored.

In [ ]:
multiframe.save_json("test.json")
mf = multiframe.load_json("test.json")